# 00 — Download RFMiD

Downloads the **Retinal Fundus Multi-disease Image Dataset (RFMiD)** used by the paper.

- Licence: **CC BY 4.0** (Pachade et al., 2021). Verify on [IEEE DataPort](https://ieee-dataport.org/open-access/retinal-fundus-multi-disease-image-dataset-rfmid).
- Preferred mirror: Kaggle `andrewmvd/retinal-disease-classification` via the **Kaggle API** and *your* `kaggle.json` (never embed a token in the notebook).
- Fallback: manual download + point `RETINA_DATA_ROOT` at the folder.

Expected layout after download:

```
$RETINA_DATA_ROOT/
  Training_Set/     (~1920 PNGs + RFMiD_Training_Labels.csv)
  Evaluation_Set/   (~640)
  Test_Set/         (~640)
```


In [ ]:
# --- cell 1: works on Colab and locally ---
from pathlib import Path
import os, sys, subprocess

IN_COLAB = False
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    pass

if IN_COLAB:
    # clone (or pull) the public repo; replace USER/REPO after you publish
    REPO_URL = os.environ.get("RETINA_REPO_URL", "https://github.com/TODO_USER/retina_multilabel.git")
    ROOT = Path("/content/retina_multilabel")
    if not (ROOT / "pyproject.toml").exists():
        subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(ROOT)])
    else:
        subprocess.call(["git", "-C", str(ROOT), "pull", "--ff-only"])
    os.chdir(ROOT)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", "."])
else:
    # local: walk up until we find the repo root
    HERE = Path.cwd().resolve()
    ROOT = HERE
    for cand in [HERE, *HERE.parents]:
        if (cand / "pyproject.toml").exists() and (cand / "scripts" / "01_build_taxonomies.py").exists():
            ROOT = cand
            break
    os.chdir(ROOT)
    if str(ROOT / "src") not in sys.path:
        sys.path.insert(0, str(ROOT / "src"))

print("IN_COLAB", IN_COLAB, "ROOT", ROOT)

In [ ]:
# --- locate / set RETINA_DATA_ROOT ---
from pathlib import Path
import os

def find_rfmid(start: Path) -> Path | None:
    """Return a directory that already contains the three RFMiD splits."""
    candidates = []
    env = os.environ.get("RETINA_DATA_ROOT")
    if env:
        candidates.append(Path(env))
    candidates += [
        ROOT / "data" / "rfmid",
        ROOT.parent / "RFMiD",
        ROOT.parent,  # e.g. .../Retina with Test_Set beside the repo
        Path.home() / "datasets" / "RFMiD",
    ]
    for c in candidates:
        if not c:
            continue
        c = c.resolve()
        # direct
        if all((c / s).is_dir() for s in ("Training_Set", "Evaluation_Set", "Test_Set")):
            return c
        # nested one level (Kaggle zip quirks / Test_Set/Test_Set)
        for child in [c, *list(c.glob("*"))[:20]]:
            if child.is_dir() and all((child / s).is_dir() for s in ("Training_Set", "Evaluation_Set", "Test_Set")):
                return child
    return None

EXISTING = find_rfmid(ROOT)
OUT = Path(os.environ.get("RETINA_DATA_ROOT") or (ROOT / "data" / "rfmid")).resolve()
print("existing RFMiD:", EXISTING)
print("download target OUT:", OUT)

In [ ]:
# --- download via Kaggle API (uses ~/.kaggle/kaggle.json or Colab secret) ---
# No tokens are hardcoded. Put kaggle.json in ~/.kaggle/ (chmod 600) or
# on Colab: userdata.get("KAGGLE_USERNAME") + userdata.get("KAGGLE_KEY").
import json, os, shutil, subprocess, sys, zipfile
from pathlib import Path

DATASET = "andrewmvd/retinal-disease-classification"
SKIP_IF_PRESENT = True

def ensure_kaggle_credentials() -> None:
    kg = Path.home() / ".kaggle" / "kaggle.json"
    if kg.is_file():
        print("using", kg)
        return
    # Colab secrets
    user = key = None
    try:
        from google.colab import userdata
        user = userdata.get("KAGGLE_USERNAME")
        key = userdata.get("KAGGLE_KEY")
    except Exception:
        pass
    user = user or os.environ.get("KAGGLE_USERNAME")
    key = key or os.environ.get("KAGGLE_KEY")
    if not (user and key):
        raise FileNotFoundError(
            "Missing Kaggle credentials.\n"
            "1) Create an API token at https://www.kaggle.com/settings → kaggle.json\n"
            "2) Place it at ~/.kaggle/kaggle.json  (or set KAGGLE_USERNAME + KAGGLE_KEY)\n"
            "3) Or skip this cell and use the manual-download cell below."
        )
    kg.parent.mkdir(parents=True, exist_ok=True)
    kg.write_text(json.dumps({"username": user, "key": key}), encoding="utf-8")
    try:
        kg.chmod(0o600)
    except Exception:
        pass
    print("wrote", kg, "from env/secrets")

def looks_complete(root: Path) -> bool:
    return all((root / s).is_dir() for s in ("Training_Set", "Evaluation_Set", "Test_Set"))

DATA_ROOT = EXISTING
if DATA_ROOT is not None and SKIP_IF_PRESENT:
    print("SKIP download; already have", DATA_ROOT)
else:
    try:
        ensure_kaggle_credentials()
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "kaggle"])
        OUT.mkdir(parents=True, exist_ok=True)
        cmd = ["kaggle", "datasets", "download", "-d", DATASET, "-p", str(OUT), "--unzip"]
        print(" ".join(cmd), flush=True)
        subprocess.check_call(cmd)
        DATA_ROOT = find_rfmid(OUT) or OUT
    except Exception as e:
        print("Kaggle download failed / skipped:", e)
        print("Use the manual-download cell next.")
        DATA_ROOT = EXISTING

if DATA_ROOT is not None:
    os.environ["RETINA_DATA_ROOT"] = str(DATA_ROOT)
    print("RETINA_DATA_ROOT =", DATA_ROOT)

In [ ]:
# --- manual-download fallback ---
# 1. Download RFMiD from IEEE DataPort (or the Kaggle page) as a zip.
# 2. Unzip so that Training_Set / Evaluation_Set / Test_Set are visible.
# 3. Set the path below and run this cell.

MANUAL_ROOT = None  # e.g. Path(r"D:/datasets/RFMiD")  or  ROOT.parent
if MANUAL_ROOT is not None:
    MANUAL_ROOT = Path(MANUAL_ROOT).resolve()
    assert all((MANUAL_ROOT / s).is_dir() for s in ("Training_Set", "Evaluation_Set", "Test_Set")), MANUAL_ROOT
    import os
    os.environ["RETINA_DATA_ROOT"] = str(MANUAL_ROOT)
    DATA_ROOT = MANUAL_ROOT
    print("RETINA_DATA_ROOT =", DATA_ROOT)
else:
    print("MANUAL_ROOT not set — skipping.")

In [ ]:
# --- verify folder structure + image counts ---
from pathlib import Path
import os

DATA_ROOT = Path(os.environ.get("RETINA_DATA_ROOT") or "") if os.environ.get("RETINA_DATA_ROOT") else None
# also accept a partial local mirror (e.g. only Test_Set next to the repo) for a soft check
soft = ROOT.parent
report = []

def count_pngs(p: Path) -> int:
    if not p.is_dir():
        return 0
    return sum(1 for _ in p.rglob("*.png")) + sum(1 for _ in p.rglob("*.JPG")) + sum(1 for _ in p.rglob("*.jpg"))

targets = {
    "Training_Set": 1920,
    "Evaluation_Set": 640,
    "Test_Set": 640,
}
root = DATA_ROOT
if root is None and (soft / "Test_Set").is_dir():
    root = soft
    report.append(f"NOTE: only a partial mirror under {soft} (no full RETINA_DATA_ROOT yet)")

if root is None:
    raise SystemExit("No RFMiD root found. Run the download cell or set MANUAL_ROOT / RETINA_DATA_ROOT.")

print("Checking", root)
ok = True
for split, expect in targets.items():
    # RFMiD sometimes nests Test_Set/Test_Set/Test
    candidates = [root / split, root / split / split, root / split / split / "Test", root / split / "Training", root / split / "Validation"]
    found = next((c for c in candidates if c.is_dir()), None)
    n = count_pngs(found) if found else 0
    status = "OK" if n >= expect * 0.9 else ("PARTIAL" if n > 0 else "MISSING")
    if status == "MISSING":
        ok = False
    print(f"  {split:16s}  n_images={n:4d}  (expect ~{expect})  [{status}]  path={found}")

# label CSVs
for csv_name in ["RFMiD_Training_Labels.csv", "RFMiD_Validation_Labels.csv", "RFMiD_Testing_Labels.csv"]:
    hits = list(root.rglob(csv_name))
    print(f"  labels {csv_name}: {'FOUND ' + str(hits[0]) if hits else 'MISSING'}")

if DATA_ROOT is not None and ok:
    print("\nReady. export RETINA_DATA_ROOT=" + str(DATA_ROOT))
elif ok:
    print("\nPartial OK. Set RETINA_DATA_ROOT to the full dataset before training.")
else:
    print("\nIncomplete — finish the download before training.")